# ADRI LangChain Integration Tutorial

This tutorial demonstrates how to integrate ADRI with LangChain agents to enforce data quality standards in your AI applications.

The ADRI LangChain integration provides tools that can be used by LangChain agents to assess data quality and ensure that only high-quality data is used.

## Setup

First, let's make sure we have ADRI and LangChain installed:

In [ ]:
# Install ADRI and LangChain if not already installed
!pip install -e ".." --quiet
!pip install langchain --quiet

Now let's import the necessary modules:

In [ ]:
import os
import pandas as pd
from adri.integrations.langchain import create_adri_tool

# Import LangChain components
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI  # You'll need an OpenAI API key for this

## Creating Sample Datasets

Let's create a couple of sample datasets to demonstrate the integration - one with good quality and one with poor quality:

In [ ]:
# Create a high-quality dataset
high_quality_data = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005],
    'name': ['John Smith', 'Jane Doe', 'Bob Johnson', 'Alice Brown', 'Charlie Davis'],
    'email': ['john@example.com', 'jane@example.com', 'bob@example.com', 'alice@example.com', 'charlie@example.com'],
    'age': [32, 28, 45, 36, 51],
    'purchase_amount': [125.50, 89.99, 210.75, 55.25, 175.00]
})

# Create a low-quality dataset with missing values and inconsistencies
low_quality_data = pd.DataFrame({
    'customer_id': [2001, 2002, None, 2004, 2005],
    'name': ['John Smith', None, 'Bob Johnson', 'Alice Brown', None],
    'email': ['not-an-email', 'jane@example.com', None, 'alice@example', 'charlie'],
    'age': [132, -28, 45, None, 51],  # Age 132 is implausible, -28 is invalid
    'purchase_amount': [None, 89.99, 210.75, 55.25, -175.00]  # Negative amount is invalid
})

# Save the datasets to CSV files
high_quality_data.to_csv('high_quality_data.csv', index=False)
low_quality_data.to_csv('low_quality_data.csv', index=False)

print("High-quality dataset:")
display(high_quality_data)

print("\nLow-quality dataset:")
display(low_quality_data)

## Creating an ADRI Tool for LangChain

Now let's create an ADRI tool that can be used by LangChain agents:

In [ ]:
# Create an ADRI tool with a minimum quality score of 70
adri_tool = create_adri_tool(min_score=70)

print(f"Tool name: {adri_tool.name}")
print(f"Tool description: {adri_tool.description}")

## Using the ADRI Tool Directly

Before integrating with an agent, let's try using the ADRI tool directly:

In [ ]:
# Try with high-quality data
try:
    print("Assessing high-quality data...")
    result = adri_tool.func('high_quality_data.csv')
    print("Assessment successful!")
    print(f"Overall score: {result['overall_score']}/100")
    print(f"Readiness level: {result['readiness_level']}")
    print("Top findings:")
    for finding in result['summary_findings'][:3]:
        print(f"  - {finding}")
except ValueError as e:
    print(f"Assessment failed: {e}")

In [ ]:
# Try with low-quality data
try:
    print("Assessing low-quality data...")
    result = adri_tool.func('low_quality_data.csv')
    print("Assessment successful!")
    print(f"Overall score: {result['overall_score']}/100")
    print(f"Readiness level: {result['readiness_level']}")
    print("Top findings:")
    for finding in result['summary_findings'][:3]:
        print(f"  - {finding}")
except ValueError as e:
    print(f"Assessment failed: {e}")

## Integrating with a LangChain Agent

Now let's integrate the ADRI tool with a LangChain agent. For this tutorial, we'll simulate the agent's behavior since running a real agent requires an OpenAI API key.

In [ ]:
# This is how you would create a LangChain agent with the ADRI tool
# Uncomment this code if you have an OpenAI API key

# import os
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# 
# llm = OpenAI(temperature=0)
# agent = initialize_agent(
#     [adri_tool], 
#     llm, 
#     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True
# )
# 
# # Use the agent
# agent.run("Assess the quality of high_quality_data.csv for use in a recommendation system")

## Simulating Agent Behavior

Let's simulate how an agent would use the ADRI tool:

In [ ]:
def simulate_agent(query):
    """Simulate a LangChain agent using the ADRI tool."""
    print(f"Agent received query: {query}")
    
    # Extract the data source from the query
    if "high_quality_data.csv" in query:
        data_source = "high_quality_data.csv"
    elif "low_quality_data.csv" in query:
        data_source = "low_quality_data.csv"
    else:
        print("Agent: I don't see a specific data source in your query.")
        return
    
    print(f"Agent: I'll assess the quality of {data_source} using the ADRI tool.")
    
    try:
        # Use the ADRI tool
        result = adri_tool.func(data_source)
        
        print(f"Agent: I've assessed the data quality of {data_source}.")
        print(f"Agent: The overall quality score is {result['overall_score']}/100.")
        print(f"Agent: The readiness level is '{result['readiness_level']}'.")
        print("Agent: Here are the key findings:")
        for finding in result['summary_findings'][:3]:
            print(f"Agent:   - {finding}")
            
        if "recommendation" in query.lower():
            if result['overall_score'] >= 70:
                print("Agent: Based on the high quality score, this data is suitable for a recommendation system.")
            else:
                print("Agent: Based on the low quality score, this data is NOT suitable for a recommendation system.")
                print("Agent: Here are some recommendations to improve the data:")
                for rec in result['summary_recommendations'][:3]:
                    print(f"Agent:   - {rec}")
    except ValueError as e:
        print(f"Agent: I encountered an issue with the data: {e}")
        print("Agent: I recommend improving the data quality before using it.")

In [ ]:
# Simulate agent with high-quality data
simulate_agent("Assess the quality of high_quality_data.csv for use in a recommendation system")

In [ ]:
# Simulate agent with low-quality data
simulate_agent("Assess the quality of low_quality_data.csv for use in a recommendation system")

## Creating a Tool Without a Minimum Score

You can also create an ADRI tool without a minimum score requirement, which will assess the data quality but not enforce a minimum standard:

In [ ]:
# Create an ADRI tool without a minimum score
assessment_tool = create_adri_tool()

# Try with low-quality data
try:
    print("Assessing low-quality data without minimum score requirement...")
    result = assessment_tool.func('low_quality_data.csv')
    print("Assessment successful!")
    print(f"Overall score: {result['overall_score']}/100")
    print(f"Readiness level: {result['readiness_level']}")
    print("Top findings:")
    for finding in result['summary_findings'][:3]:
        print(f"  - {finding}")
except ValueError as e:
    print(f"Assessment failed: {e}")

## Conclusion

The ADRI LangChain integration provides a powerful way to enforce data quality standards in your LangChain agents. By using the ADRI tool, you can ensure that your agents only work with data that meets your quality requirements.

Key benefits:

1. **Seamless Integration**: The ADRI tool integrates easily with LangChain agents
2. **Customizable Thresholds**: You can set different minimum quality scores for different tools
3. **Detailed Assessment**: The tool provides comprehensive data quality assessments
4. **Quality Enforcement**: The tool can block the use of low-quality data

For more information, see the [INTEGRATIONS.md](../INTEGRATIONS.md) guide.

## Cleanup

Let's clean up the sample files we created:

In [ ]:
# Remove the sample files
os.remove('high_quality_data.csv')
os.remove('low_quality_data.csv')
print("Sample files removed.")